# llminfer README Walkthrough (Colab)

This notebook mirrors the README flow for `llminfer`: quick start, streaming, batching, prefix KV cache reuse, benchmarking, backend comparison, and CLI.

## 0) Runtime setup
In Colab, select **Runtime -> Change runtime type -> GPU** before running cells.

In [ ]:
!nvidia-smi

In [ ]:
import platform
import torch

print('Python:', platform.python_version())
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('CUDA version:', torch.version.cuda)

if not torch.cuda.is_available():
    raise RuntimeError('GPU not available. Switch runtime to GPU and retry.')

## 1) Clone and install llminfer
Update `REPO_URL` to your repo URL.

In [ ]:
REPO_URL = 'https://github.com/<your-user>/<your-repo>.git'  # TODO
TARGET_DIR = '/content/llminfer'

import os
import shutil

if os.path.exists(TARGET_DIR):
    shutil.rmtree(TARGET_DIR)

!git clone {REPO_URL} {TARGET_DIR}
%cd /content/llminfer
!pip -q install -U pip
!pip -q install -e .

## 2) Optional Hugging Face login
Required for gated/private models only.

In [ ]:
# from huggingface_hub import notebook_login
# notebook_login()

## 3) Quick Start (README-style)
The README uses `facebook/opt-1.3b`. You can switch to `facebook/opt-125m` if memory is tight.

In [ ]:
from llminfer import InferenceEngine, EngineConfig
from llminfer.config import Backend, QuantConfig, QuantMode

MODEL = 'facebook/opt-1.3b'  # swap to 'facebook/opt-125m' if needed

# Eager inference, no quantization
engine = InferenceEngine(EngineConfig(model_name=MODEL))
engine.load()

result = engine.run('Explain attention mechanisms.', max_new_tokens=96)
print(result.generated_text)
print(f"Latency: {result.stats.total_latency_ms:.0f} ms | {result.stats.throughput_tokens_per_sec:.1f} tok/s")

### 3.1) 4-bit quantization (README equivalent)
This often helps fit larger models on Colab GPUs.

In [ ]:
cfg_4bit = EngineConfig(
    model_name=MODEL,
    quant=QuantConfig(mode=QuantMode.NF4, double_quant=True),
)
engine_4bit = InferenceEngine(cfg_4bit)
engine_4bit.load()

res_4bit = engine_4bit.run('Give me a concise definition of QLoRA.', max_new_tokens=80)
print(res_4bit.generated_text)
engine_4bit.unload()

### 3.2) Streaming (README equivalent)

In [ ]:
print('Streaming:', end=' ', flush=True)
for chunk in engine.stream('Tell me a short story about GPUs.', max_new_tokens=80):
    if not chunk.is_final:
        print(chunk.token, end='', flush=True)
    else:
        print()
        if chunk.stats:
            print('Final stats:', chunk.stats)

### 3.3) Batch inference (README equivalent)

In [ ]:
results = engine.run_batch([
    'What is RLHF?',
    'Explain LoRA.'
], max_new_tokens=64)

for i, r in enumerate(results, start=1):
    print(f'Result {i}:', r.generated_text)

### 3.4) Prefix KV cache reuse (README equivalent)

In [ ]:
from llminfer.kv_cache import KVCacheManager

sys_prompt = 'You are a helpful AI assistant.'
key = KVCacheManager.hash_prefix(sys_prompt)

r1 = engine.run(sys_prompt + '\n\nUser: What is BERT?', max_new_tokens=64, prefix_key=key)
r2 = engine.run(sys_prompt + '\n\nUser: What is GPT-2?', max_new_tokens=64, prefix_key=key)

print('Call 1 cache_hit:', r1.stats.cache_hit)
print('Call 2 cache_hit:', r2.stats.cache_hit)
print('Cache stats:', engine.cache_stats())

## 4) Benchmarking (README section)

In [ ]:
from llminfer import Benchmarker

bm = Benchmarker(engine)
bench = bm.run(batch_sizes=[1, 2, 4, 8], num_runs=5, max_new_tokens=64)
bench.print_summary()
bench.plot('/content/bench_readme.png')

In [ ]:
from IPython.display import Image, display
display(Image('/content/bench_readme.png'))

## 5) Backend comparison (README section)
Start with eager + compiled in Colab. vLLM is optional and may not install on all images.

In [ ]:
from llminfer.benchmark import BackendComparison
from llminfer.config import Backend

cmp = BackendComparison(
    model_name=MODEL,
    backends=[Backend.EAGER, Backend.COMPILED],
)
cmp_results = cmp.run(batch_sizes=[1, 2, 4, 8], num_runs=3, max_new_tokens=64)
cmp.print_table(cmp_results)
cmp.plot(cmp_results, '/content/comparison_readme.png')

In [ ]:
import os
from IPython.display import Image, display

if os.path.exists('/content/comparison_readme.png'):
    display(Image('/content/comparison_readme.png'))
else:
    print('Comparison plot not found.')

### 5.1) Optional vLLM backend
Only run this if your Colab image supports vLLM installation.

In [ ]:
# !pip install -e '.[vllm]'
# from llminfer.config import Backend
# cmp_v = BackendComparison(model_name=MODEL, backends=[Backend.EAGER, Backend.VLLM])
# results_v = cmp_v.run(batch_sizes=[1,2,4], num_runs=3, max_new_tokens=64)
# cmp_v.print_table(results_v)

## 6) CLI commands (README section)

In [ ]:
!llminfer run "Tell me about GPUs" --model facebook/opt-125m

In [ ]:
!llminfer stream "Explain backpropagation" --model facebook/opt-125m --quant none --max-tokens 64

In [ ]:
!llminfer bench --model facebook/opt-125m --batch-sizes 1,2,4,8 --runs 5 --plot /content/bench_cli.png

In [ ]:
!llminfer compare --model facebook/opt-125m --backends eager,compiled --plot /content/compare_cli.png

In [ ]:
!llminfer info --model facebook/opt-125m --backend compiled

## 7) Cleanup

In [ ]:
engine.unload()